> **Executed output.** Exported from Databricks job `northbay-end-to-end` run 59476289597609, task `land_raw_extracts` (task run 573139648151470), started 2026-11-08 08:23 UTC, duration 21s, result **SUCCESS**, serverless compute.


# 01 · Land raw source-system extracts (synthetic)

NorthBay Foods' planning data lives in five systems. This notebook simulates their nightly extracts and lands
them as raw files in a Unity Catalog Volume, which the Lakeflow pipeline ingests incrementally with Auto Loader.

| System | Extract | Format |
|---|---|---|
| ERP | Finished-goods plants: capacity, margin, price | CSV |
| SRM | Supplier & material masters, expected recovery times | CSV |
| PLM | Approved supply links (who supplies what to whom) | JSON lines |
| WMS | Weekly on-hand inventory snapshots | JSON lines |
| APS | 13-week demand forecast | CSV |
| Risk | Named multi-node disruption scenarios | CSV |

The data is fully synthetic (fixed seed). A few realistic data-quality issues are injected on purpose
(a duplicated supplier record, a negative inventory scan, a link to a supplier missing from SRM) so the
pipeline's expectations have something to catch.

In [ ]:
dbutils.widgets.text("catalog", "classic_stable_z57t9z_catalog")
dbutils.widgets.text("schema", "febar_supply_chain")
dbutils.widgets.text("serving_schema", "northbay_serving")
catalog = dbutils.widgets.get("catalog")
schema = dbutils.widgets.get("schema")
serving_schema = dbutils.widgets.get("serving_schema")
raw_root = f"/Volumes/{catalog}/{schema}/raw_landing"
print(raw_root)

/Volumes/classic_stable_z57t9z_catalog/febar_supply_chain/raw_landing


In [ ]:
import os
import sys

sys.path.append(os.path.abspath(".."))  # src/ -> northbay_data package
from northbay_data.synthetic import generate, PRODUCTS, SUPPLIERS

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.{schema} "
          "COMMENT 'NorthBay Foods supply network resilience data product: raw volume, bronze/silver/gold, agent'")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.{serving_schema} "
          "COMMENT 'Lakebase synced tables: operational serving copies of the gold tables'")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {catalog}.{schema}.raw_landing COMMENT 'Raw landing zone for source-system extracts'")

files = generate()
for rel_path, content in files.items():
    path = f"{raw_root}/{rel_path}"
    os.makedirs(os.path.dirname(path), exist_ok=True)
    with open(path, "w") as fh:
        fh.write(content)

print(f"Network: {len(PRODUCTS)} finished-goods plants, "
      f"{sum(1 for s in SUPPLIERS if s[2] == 2)} Tier-2 and {sum(1 for s in SUPPLIERS if s[2] == 3)} Tier-3 suppliers")
for rel_path in sorted(files):
    print(f"{len(files[rel_path].splitlines()):>4} lines  {raw_root}/{rel_path}")

Network: 5 finished-goods plants, 10 Tier-2 and 14 Tier-3 suppliers
  66 lines  /Volumes/classic_stable_z57t9z_catalog/febar_supply_chain/raw_landing/aps/demand_forecast/forecast_20261102.csv
   6 lines  /Volumes/classic_stable_z57t9z_catalog/febar_supply_chain/raw_landing/erp/products/products_20261102.csv
  36 lines  /Volumes/classic_stable_z57t9z_catalog/febar_supply_chain/raw_landing/plm/supply_links/supply_links.json
   5 lines  /Volumes/classic_stable_z57t9z_catalog/febar_supply_chain/raw_landing/risk/scenarios/scenarios.csv
  13 lines  /Volumes/classic_stable_z57t9z_catalog/febar_supply_chain/raw_landing/srm/materials/materials.csv
  26 lines  /Volumes/classic_stable_z57t9z_catalog/febar_supply_chain/raw_landing/srm/suppliers/suppliers_20261102.csv
  29 lines  /Volumes/classic_stable_z57t9z_catalog/febar_supply_chain/raw_landing/wms/inventory/inventory_2026-10-12.json
  29 lines  /Volumes/classic_stable_z57t9z_catalog/febar_supply_chain/raw_landing/wms/inventory/inventory_2026-1

Peek at two raw extracts as the source systems deliver them.

In [ ]:
print(files["erp/products/products_20261102.csv"])
print("\n".join(files["plm/supply_links/supply_links.json"].splitlines()[:5]))

product_id,product_name,brand,category,plant_name,plant_city,plant_state,capacity_cases_per_week,margin_usd_per_case,price_usd_per_case,updated_at
FG01,Kettle Chips 8oz,Crunch Co.,Snacks,Fresno Snacks Plant,Fresno,CA,67429,6.4,21.0,2026-11-02
FG02,Morning Oats Cereal 18oz,Morning Oats,Cereal,Cedar Rapids Plant,Cedar Rapids,IA,54022,7.2,28.5,2026-11-02
FG03,SunValley Orange Drink 1L,SunValley,Beverages,Lakeland Bottling,Lakeland,FL,42082,5.1,18.0,2026-11-02
FG04,Fizz Cola 2L,Fizz,Beverages,Atlanta Bottling,Atlanta,GA,101760,3.8,14.5,2026-11-02
FG05,Trail Granola Bar 6ct,Trail Mix Co.,Snacks,Battle Creek Plant,Battle Creek,MI,33615,8.4,24.0,2026-11-02

{"source_id": "S201", "target_id": "FG01", "material_id": "M-SEAS", "approved": true, "approved_on": "2025-03-01"}
{"source_id": "S205", "target_id": "FG01", "material_id": "M-FILM", "approved": true, "approved_on": "2025-03-01"}
{"source_id": "S206", "target_id": "FG01", "material_id": "M-FILM", "approved": true, "approved_on": "2025-03-0